# Veltis — deux collections Qdrant : 544 vs 100 tokens

**Objectif :** conserver les 84 chunks historiques (chunker `max_tokens=544`) et fabriquer une nouvelle variante (chunker `max_tokens=100`) à partir des **mêmes 13 exports Docling**. Les deux variantes sont stockées dans **une seule instance Qdrant**, dans **deux collections différentes**.

| Version | Collection | Provenance | Embeddings |
|---|---|---|---|
| Historique | `veltis_dense_544` | `veltis_chunks.json` (84 passages) | `veltis_embeddings.npz` (84 × 1024) |
| Expérimentale | `veltis_dense_100` | 13 JSON Docling, nouveau découpage | Nouveaux vecteurs BGE-M3 |

**Règles :** ne supprimer aucune collection; ne pas retraiter les PDF; aucun appel à BGE-M3 pour importer la version 544. Une cellule à la fois. Le réglage du chunker est une **cible**, et le texte contextualisé peut dépasser cette valeur. La comparaison ultérieure devra utiliser les mêmes questions dans les deux collections.

Le notebook original `demo_qdrant.ipynb` utilise `api_key="1234"` pour son serveur Qdrant local de formation. Ne pas réutiliser ce secret dans un dépôt public.

In [1]:
# CELLULE 1 — Imports, chemins et noms (aucune connexion réseau)
from pathlib import Path
import json
import uuid
import numpy as np
from qdrant_client import QdrantClient, models

ROOT = Path('/home/bm/formation/RAG/cours-rag/drive-download-20261008T064617Z-1-001/rag/demo')
EXPORT = ROOT / 'export_folder'
MODEL_PATH = Path('/home/bm/models/rag/embedder/bge-m3')
OLD_VECTORS = EXPORT / 'veltis_embeddings.npz'
OLD_CHUNKS = EXPORT / 'veltis_chunks.json'
NEW_VECTORS = EXPORT / 'veltis_embeddings_100.npz'
NEW_CHUNKS = EXPORT / 'veltis_chunks_100.json'
COLL_544 = 'veltis_dense_544'
COLL_100 = 'veltis_dense_100'

# Les exports Docling sont nommés source.ext.json (et non veltis_chunks.json).
DOC_EXTENSIONS = {'.pdf', '.docx', '.html', '.pptx'}
source_files = sorted(f for f in EXPORT.glob('*.json')
                      if Path(f.stem).suffix.lower() in DOC_EXTENSIONS)

assert len(source_files) == 13, f'13 exports attendus, trouvé {len(source_files)} : {[f.name for f in source_files]}'
assert OLD_VECTORS.is_file() and OLD_CHUNKS.is_file(), 'Fichiers 544 manquants'
print('Exports Docling :', len(source_files))
print('Fichiers 544 : OK')
print('Collections cibles :', COLL_544, '|', COLL_100)

Exports Docling : 13
Fichiers 544 : OK
Collections cibles : veltis_dense_544 | veltis_dense_100


In [2]:
# CELLULE 2 — Connexion Qdrant (authentification du notebook de cours)
qclient = QdrantClient(url='http://127.0.0.1:6333', api_key='1234')
print('Qdrant connecté. Collections existantes :')
for c in qclient.get_collections().collections:
    print(' -', c.name)

Qdrant connecté. Collections existantes :
 - demo_collection
 - veltis_dense_100
 - veltis_dense_544


/tmp/ipykernel_2253595/1254527351.py:2: UserWarning: Api key is used with an insecure connection.
  qclient = QdrantClient(url='http://127.0.0.1:6333', api_key='1234')


In [3]:
# CELLULE 3 — Créer les deux collections, sans effacer une collection existante
# Uniquement dense : les données sauvegardées hier ne contiennent pas de vecteurs sparse.
for name in (COLL_544, COLL_100):
    if not qclient.collection_exists(name):
        qclient.create_collection(
            collection_name=name,
            vectors_config={'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE)}
        )
        print('Créée :', name)
    else:
        print('Déjà présente (non modifiée) :', name)

    config = qclient.get_collection(name).config.params.vectors
    assert 'dense' in config and config['dense'].size == 1024, f'Configuration incompatible : {name}'
print('Les deux collections sont configurées avec dense/1024/cosinus.')

Déjà présente (non modifiée) : veltis_dense_544
Déjà présente (non modifiée) : veltis_dense_100
Les deux collections sont configurées avec dense/1024/cosinus.


In [4]:
# CELLULE 4 — Charger et valider les données sauvegardées (version 544)
with np.load(OLD_VECTORS) as saved:
    vectors_544 = saved['embeddings']
with OLD_CHUNKS.open(encoding='utf-8') as f:
    chunks_544 = json.load(f)
assert vectors_544.shape == (84, 1024)
assert len(chunks_544) == 84
assert np.isfinite(vectors_544).all()
assert all(item.get('source') and item.get('text') for item in chunks_544)
print('Ancienne version :', len(chunks_544), 'chunks | vecteurs :', vectors_544.shape)
print('Exemple #69 :', chunks_544[69]['source'])
print('Exemple #9 :', chunks_544[9]['source'])
print('Exemple #9 :', chunks_544[60]['source'])

Ancienne version : 84 chunks | vecteurs : (84, 1024)
Exemple #69 : pv_conseil_administration_2026-03-12_scan.pdf
Exemple #9 : guide_onboarding.pdf
Exemple #9 : presentation_veltis.pdf


In [5]:
# CELLULE 5 — Injecter les 84 anciens points dans veltis_dense_544
# UUID stables : répéter la cellule met à jour les mêmes points, sans doublon.
points_544 = []
for i, (chunk, vector) in enumerate(zip(chunks_544, vectors_544)):
    point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'veltis-544/{i}'))
    points_544.append(models.PointStruct(
        id=point_id,
        vector={'dense': vector.tolist()},
        payload={
            'dataset': 'veltis', 'chunk_profile': '544',
            'chunker_target_tokens': 544,
            'global_index': i,
            'file_name': chunk['source'],
            'text': chunk['text']
        }
    ))
res = qclient.upsert(collection_name=COLL_544, points=points_544, wait=True)
assert str(res.status).lower().endswith('completed')
check = qclient.retrieve(COLL_544, ids=[points_544[69].id], with_payload=True)
assert len(check) == 1 and check[0].payload['global_index'] == 69
print(COLL_544, ':', len(points_544), 'points transmis ; source #69 :', check[0].payload['file_name'])
check = qclient.retrieve(COLL_544, ids=[points_544[9].id], with_payload=True)
assert len(check) == 1 and check[0].payload['global_index'] == 9
print(COLL_544, ':', len(points_544), 'points transmis ; source #9 :', check[0].payload['file_name'])
print('Total dans cette collection :', qclient.count(COLL_544, exact=True).count)

veltis_dense_544 : 84 points transmis ; source #69 : pv_conseil_administration_2026-03-12_scan.pdf
veltis_dense_544 : 84 points transmis ; source #9 : guide_onboarding.pdf
Total dans cette collection : 84


## Nouvelle variante 100 tokens

Le **même modèle BGE-M3** et les **mêmes exports Docling** sont utilisés pour limiter les variables expérimentales. Les chunks 100 ne remplacent jamais ceux de 544. Les embeddings seront enregistrés dans deux **nouveaux** fichiers explicites (`veltis_*_100.*`).

In [24]:
# CELLULE 6 — Découper les 13 documents avec une cible de 100 tokens
from docling_core.types.doc import DoclingDocument
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
chunker_100 = HybridChunker(tokenizer=HuggingFaceTokenizer(tokenizer=tokenizer, max_tokens=100))
chunks_100 = []
for file in source_files:
    document = DoclingDocument.load_from_json(file)
    pieces = list(chunker_100.chunk(document))
    for local_index, piece in enumerate(pieces):
        chunks_100.append({
            'source': file.name.removesuffix('.json'),
            'local_index': local_index,
            'text': chunker_100.contextualize(piece)
        })
    print(f'{file.name}: {len(pieces)} chunks')
print('TOTAL chunks 100 :', len(chunks_100))

grille_salariale_2026.pdf.json: 12 chunks
guide_onboarding.pdf.json: 14 chunks
intranet_actualites.html.json: 5 chunks
intranet_faq_rh.html.json: 8 chunks
newsletter_le_fil_2026-02.pdf.json: 11 chunks
organigramme_et_processus_production.pdf.json: 2 chunks
plan_strategique_2026-2028.pptx.json: 5 chunks
politique_teletravail_2023.docx.json: 7 chunks
politique_teletravail_2025.docx.json: 8 chunks
politique_teletravail_2025.pdf.json: 8 chunks
presentation_veltis.pdf.json: 11 chunks
pv_conseil_administration_2026-03-12_scan.pdf.json: 8 chunks
rapport_annuel_2025.pdf.json: 30 chunks
TOTAL chunks 100 : 129


In [25]:
# CELLULE 7 — Mesurer les tailles réelles et enregistrer les nouveaux chunks
texts_100 = [item['text'] for item in chunks_100]
tokens_100 = [len(tokenizer.encode(text)) for text in texts_100]
assert all(text.strip() for text in texts_100), 'Chunk vide détecté'
print('Chunks :', len(chunks_100))
print('Tokens min / médiane / max :', min(tokens_100), float(np.median(tokens_100)), max(tokens_100))
print('Chunks > 100 :', sum(t > 100 for t in tokens_100))
print('Chunks > 128 :', sum(t > 128 for t in tokens_100))
# Ne pas déclarer 100 tokens comme limite stricte : contextualisation et tableaux peuvent dépasser la cible.
with NEW_CHUNKS.open('w', encoding='utf-8') as f:
    json.dump(chunks_100, f, ensure_ascii=False, indent=2)
print('Métadonnées enregistrées :', NEW_CHUNKS.name)

Chunks : 129
Tokens min / médiane / max : 9 63.0 110
Chunks > 100 : 31
Chunks > 128 : 0
Métadonnées enregistrées : veltis_chunks_100.json


In [26]:
# CELLULE 8 — Vérifier le serveur BGE-M3 avant de calculer de nouveaux vecteurs
from openai import OpenAI
embedder = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='not-needed', timeout=120.0)
print('Modèles disponibles :', [m.id for m in embedder.models.list().data])
# Si erreur de connexion : relancer vLLM sur Z4 avant de continuer.

Modèles disponibles : ['bge-m3']


In [27]:
# CELLULE 9 — Générer les nouveaux embeddings par lots de 16
# Attention : c'est le seul passage qui nécessite le modèle BGE-M3 en cours d'exécution.
batch_size = 16
batches = []
for start in range(0, len(texts_100), batch_size):
    batch = texts_100[start:start + batch_size]
    response = embedder.embeddings.create(model='bge-m3', input=batch)
    ordered = sorted(response.data, key=lambda item: item.index)
    assert len(ordered) == len(batch)
    batches.extend(item.embedding for item in ordered)
    print('Embeddings :', len(batches), '/', len(texts_100))
vectors_100 = np.asarray(batches, dtype=np.float32)
assert vectors_100.shape == (len(chunks_100), 1024)
assert np.isfinite(vectors_100).all()
np.savez_compressed(NEW_VECTORS, embeddings=vectors_100)
print('Enregistré :', NEW_VECTORS.name, '| shape :', vectors_100.shape)

Embeddings : 16 / 129
Embeddings : 32 / 129
Embeddings : 48 / 129
Embeddings : 64 / 129
Embeddings : 80 / 129
Embeddings : 96 / 129
Embeddings : 112 / 129
Embeddings : 128 / 129
Embeddings : 129 / 129
Enregistré : veltis_embeddings_100.npz | shape : (129, 1024)


In [28]:
# CELLULE 10 — Importer la variante 100 dans sa collection distincte
points_100 = []
for i, (chunk, vector) in enumerate(zip(chunks_100, vectors_100)):
    point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'veltis-100/{i}'))
    points_100.append(models.PointStruct(
        id=point_id,
        vector={'dense': vector.tolist()},
        payload={
            'dataset': 'veltis', 'chunk_profile': '100',
            'chunker_target_tokens': 100,
            'actual_token_count': tokens_100[i],
            'global_index': i, 'local_index': chunk['local_index'],
            'file_name': chunk['source'], 'text': chunk['text']
        }
    ))
# Lots : plus robuste si la nouvelle variante contient de nombreux petits chunks.
for start in range(0, len(points_100), 64):
    res = qclient.upsert(collection_name=COLL_100, points=points_100[start:start+64], wait=True)
    assert str(res.status).lower().endswith('completed')
print('Points version 100 transmis :', len(points_100))
print('Total dans', COLL_100, ':', qclient.count(COLL_100, exact=True).count)

Points version 100 transmis : 129
Total dans veltis_dense_100 : 129


In [29]:
# CELLULE 11 — Vérification comparative (sans rechercher encore)
for name, target in [(COLL_544, 544), (COLL_100, 100)]:
    count = qclient.count(name, exact=True).count
    sample, _ = qclient.scroll(name, limit=1, with_payload=True, with_vectors=False)
    assert sample and sample[0].payload['chunk_profile'] == str(target)
    print(f'{name}: {count} points | profil = {sample[0].payload["chunk_profile"]} tokens')
assert qclient.count(COLL_544, exact=True).count == 84
assert qclient.count(COLL_100, exact=True).count == len(chunks_100)
print('Deux collections distinctes : VALIDÉ')

veltis_dense_544: 84 points | profil = 544 tokens
veltis_dense_100: 129 points | profil = 100 tokens
Deux collections distinctes : VALIDÉ


## Après-midi : recherche indépendante

Prochain exercice, sans solution complète fournie d'avance : interroger les deux collections avec la même question et le même modèle d'embeddings. Comparer rang du passage correct, Hit@1/Hit@5 et caractère complet de la réponse. Commencer par les trois questions d'accueil et celle du dividende. Préserver les deux variantes pour pouvoir refaire les mesures.